# Scenario 1: Customer Support Resolution Agent

## 1. In One Minute

**What it is.** An Agent SDK agent that handles returns, billing disputes, and account issues. It uses four custom MCP tools: `get_customer`, `lookup_order`, `process_refund`, `escalate_to_human`. The target is 80%+ first-contact resolution, while still knowing when to hand off to a human.

**Domains hit:** D1 Agentic Architecture, D2 Tool Design and MCP, D5 Context Management and Reliability.

**Task statements:** TS 1.1 (agent loop), 1.4 (prerequisite gates), 1.5 (hooks), 2.1 (tool descriptions), 2.2 (MCP errors), 2.3 (tool distribution), 5.1 (context and case facts), 5.2 (escalation).

**Top 3 traps**
1. Using a prompt or few-shot examples to enforce a rule that must hold 100% of the time (identity check before a refund).
2. Using self-reported confidence, sentiment, or a separate classifier to decide when to escalate.
3. Building a routing classifier or merging tools before fixing weak tool descriptions.

## 2. Architecture Diagram

```mermaid
---
config:
  flowchart:
    htmlLabels: false
---
flowchart TD
    User["Customer message"] --> Loop["Agent loop<br>(stop_reason decides)"]
    Loop -->|"stop_reason: tool_use"| Hook{"Code gate / hook<br>(prerequisites, refund limit)"}
    Hook -->|"Blocked"| Err["Structured error<br>(isError, errorCategory, isRetryable)"]
    Hook -->|"Allowed"| Tools["MCP tools:<br>get_customer, lookup_order,<br>process_refund, escalate_to_human"]
    Tools --> Trim["PostToolUse: trim payload<br>to the few fields needed"]
    Trim --> Facts["Case facts block<br>(customer, order, status)"]
    Err --> Loop
    Facts --> Loop
    Loop -->|"stop_reason: end_turn"| Reply["Reply to customer"]
    Hook -->|"Policy gap / human request"| Human["escalate_to_human<br>(structured handoff)"]
```

## 3. How a Good Answer Thinks

Ask these in order.

1. **What is the root cause?** Do not treat the symptom. Wrong tool picked? Look at tool descriptions first.
2. **Must it hold every time?** Money, identity, and permissions need code (gate or hook). Prompts and few-shot examples are probabilistic.
3. **Is the first step proportionate?** Cheapest fix that attacks the cause: better descriptions before classifiers, explicit criteria before extra models.
4. **Least privilege.** Give the agent only the 4-5 tools it needs.
5. **Escalation uses explicit criteria**: customer asks for a human, policy gap or exception, no progress. Never sentiment or self-reported confidence.
6. **Escalate right away** on an explicit human request. Do not probe or retry first.
7. **Several matches?** Ask for another identifier. Never guess by recency.

```mermaid
---
config:
  flowchart:
    htmlLabels: false
---
flowchart TD
    Q{"What does the question describe?"} -->|"Rule must hold every time"| Code["Programmatic gate or hook"]
    Q -->|"Agent picks the wrong tool"| Desc["Expand tool descriptions first"]
    Q -->|"Customer asks for a human"| Esc["Escalate now, with handoff"]
    Q -->|"Policy gap or exception"| Esc2["Escalate with explicit criteria"]
    Q -->|"Long session loses facts"| Facts["Case facts block + trimming"]
    Q -->|"Tool failed"| Struct["Structured error, retry only if retryable"]
```

### Model and Sampling Notes
- Use `claude-sonnet-4-5` for coordinators and reasoning, `claude-haiku-4-5` for workers and extraction.
- With extended thinking on: omit `temperature`, leave `top_k` and `top_p` unset, and use only `tool_choice` `auto` or `none` (forced or `any` returns HTTP 400). Forced-tool extraction calls must not use thinking.
- In tool loops, return thinking blocks with their signature unchanged.
- Loops end on `stop_reason` only (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, `pause_turn`, `refusal`). A max-iterations counter is a safety cap, not the stop rule.

## 4. Task Statement Map

| Task statement | How it shows up here | Trap |
|---|---|---|
| TS 1.1 Agent loop | Loop continues on `tool_use`, stops on `end_turn` | Parsing reply text to decide when to stop |
| TS 1.4 Prerequisite gates | `get_customer` must succeed before `lookup_order` or `process_refund` | Prompt rule or few-shot for a 100% rule |
| TS 1.5 Hooks | Block `process_refund` above `$500` in code; trim results in `PostToolUse` | Asking the model to remember the limit |
| TS 2.1 Tool design | `get_customer` (who) vs `lookup_order` (what) need clear boundaries | Adding a routing classifier before fixing descriptions |
| TS 2.2 MCP errors | Return `isError`, `errorCategory`, `isRetryable` | Generic "operation failed" or empty success |
| TS 2.3 Distribution | Support agent gets only its 4 tools | Giving all company tools "just in case" |
| TS 5.1 Context | Case facts block + trimmed tool output | Relying on summaries for IDs and amounts |
| TS 5.2 Escalation | Explicit criteria + few-shot for borderline cases | Sentiment score or self-reported confidence |

## 5. Worked Example

**Customer:** "My name is Alex Kim. Refund my desk order."

1. Name search finds **two** Alex Kims. The agent asks for the phone number or email. It does not pick one.
2. Customer gives `alex1@example.com`. `get_customer` succeeds, so case facts now hold `customer_id` and `is_authenticated`.
3. `lookup_order` returns 40 fields. `PostToolUse` keeps `order_id`, `status`, `total`, `items`, `days_since_delivery`.
4. Customer wants a refund of `$750`. The refund hook sees the amount is over `$500`, blocks the tool, and returns a structured error.
5. The agent calls `escalate_to_human` with case facts, actions taken, and the reason. A human finishes the case.

Why this is right: identity and limit are enforced by code, context stays small, and the escalation reason is explicit.

Try it offline: `scenarios/scenario_1_customer_support/simulation.py` (`execute_tool`, `resolve_customer_by_name`, `REFUND_APPROVAL_LIMIT_USD`).

## 7. Self-Check Checklist

- [ ] I can say why code beats prompts for identity and money.
- [ ] I know the four tools and what each one is for.
- [ ] I can list three valid escalation triggers and two invalid ones.
- [ ] I know what to keep in the case facts block and what to trim.
- [ ] I fix tool descriptions before adding classifiers.

**Memory hook:** "Money in code, humans on request, facts in a block."


In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_1_customer_support.simulation import *

print(' Scenario Workbench Initialized: Scenario 1: Customer Support Resolution Agent')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Execute Happy Path: Authentication + Return Lookup with Trimming
pipeline = CustomerSupportPipeline()

# 1. Auth
auth_res = pipeline.process_customer_turn("My email is alice@example.com")
print("Auth action:", auth_res["action_taken"])
assert pipeline.case_facts.is_authenticated is True

# 2. Lookup order
order_res = pipeline.process_customer_turn("Look up ord_9901")
print("Order action:", order_res["action_taken"])
print("Trimmed Payload:", order_res["order_data"])
assert "_internal_warehouse_id" not in order_res["order_data"]

# 3. Process refund
refund_res = pipeline.process_customer_turn("Please issue the refund.")
print("Refund action:", refund_res["action_taken"])
assert refund_res["action_taken"] == "REFUND_SUCCESS"



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Execute Edge Cases: Unauthenticated Refund Attempt & Immediate Escalation
pipeline2 = CustomerSupportPipeline()

# Edge Case 1: Unauthenticated lookup blocked
res_blocked = pipeline2.process_customer_turn("Please check ord_9901")
print("Blocked action:", res_blocked["action_taken"])
assert res_blocked["action_taken"] == "GATE_BLOCKED_UNAUTHENTICATED"

# Edge Case 2: Immediate human escalation
res_esc = pipeline2.process_customer_turn("I want to speak to a human right now.")
print("Escalation action:", res_esc["action_taken"])
assert res_esc["action_taken"] == "ESCALATED_TO_HUMAN"
assert pipeline2.escalated is True



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.


### Step 5: New behaviour - refund limit, gate error, handoff
Offline checks for the extended simulation.


In [ ]:
pipe = CustomerSupportPipeline()
blocked = pipe.execute_tool("process_refund", order_id="ord_9901")
assert blocked["isError"] is True and blocked["isRetryable"] is False
pipe.process_customer_turn("My email is bob@example.com")
res = pipe.process_customer_turn("Refund ord_9903")  # $750 > $500 limit
assert res["action_taken"] == "ESCALATED_TO_HUMAN"
assert pipe.escalation_data.escalation_type == "THRESHOLD"
print("Handoff actions:", pipe.escalation_data.actions_taken)
